In [1]:
import chess.svg
import einops
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn

from chess_gnn.models import *
from chess_gnn.utils import PGNBoardHelper, ChessPoint
from chess_gnn.tokenizers import SimpleChessTokenizer

import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objects as go
import chess
import chess.svg
import base64

In [2]:
class MaskVisualizationHelper(nn.Module):
    def __init__(self, transformer: ChessTransformer):
        super().__init__()
        self.transformer = transformer
        self.tokenizer = SimpleChessTokenizer()
        
    def get_preds(self, game_batch: dict[str, torch.Tensor]):
        batch = self.transformer.squeeze_batch(game_batch)

        # mask is shared by current and next boards
        ids_shuffle, ids_restore, len_keep = self.transformer.mask_handler.get_mask(batch['board'])
        current_board_encoded = self.transformer.encode(batch['board'], batch['whose_move'], self.transformer.current_board_cls_token, ids_shuffle, ids_restore, len_keep)
        next_board_encoded = self.transformer.encode(batch['next_board'], torch.logical_not(batch['whose_move']).long(), self.transformer.next_board_cls_token, 
                                                     ids_shuffle, ids_restore, len_keep)

        current_board_preds = self.transformer.decode(next_board_encoded['cls'], current_board_encoded['decoder_in'])
        next_board_preds = self.transformer.decode(current_board_encoded['cls'], next_board_encoded['decoder_in'])
        
        current_board_preds = einops.rearrange(current_board_preds, 'b c l -> b l c').argmax(dim=-1)
        next_board_preds = einops.rearrange(next_board_preds, 'b c l -> b l c').argmax(dim=-1)
        
        mask = torch.zeros_like(batch['board'])
        mask[..., len_keep:] = 1
        mask = torch.gather(mask, dim=1, index=ids_restore)
        
        full_current_pred = batch['board'].clone()
        full_current_pred = torch.gather(full_current_pred, dim=1, index=ids_shuffle)
        full_current_pred[..., len_keep:] = current_board_preds
        full_current_pred = torch.gather(full_current_pred, dim=1, index=ids_restore)
        
        full_next_pred = batch['next_board'].clone()
        full_next_pred = torch.gather(full_next_pred, dim=1, index=ids_shuffle)
        full_next_pred[..., len_keep:] = next_board_preds
        full_next_pred = torch.gather(full_next_pred, dim=1, index=ids_restore)
        
        return mask, full_current_pred, full_next_pred    
        

In [3]:
class ChessBoardArray:
    def __init__(self, board_array: np.array):
        self.board_array = board_array
        if len(self.board_array) != 64:
            raise ValueError("Board array must be of length 64")
        self.tokenizer = SimpleChessTokenizer()
    
    def __len__(self):
        return len(self.board_array)
    
    def __getitem__(self, item):
        return self.board_array[item]
    
    def to_pieces(self):
        untokenized = self.tokenizer.untokenize(self.board_array)
        untokenized = [token if token != '.' else None for token in untokenized]
        return untokenized

In [4]:
def board_to_svg_image(board: ChessBoardArray):
    images = []
    pieces = board.to_pieces()
    for idx in range(64):
        piece = pieces[idx]
        point = ChessPoint.from_1d(idx)
        if piece:
            svg = chess.svg.piece(chess.Piece.from_symbol(piece))
            svg_bytes = svg.encode('utf-8')
            uri = f"data:image/svg+xml;base64,{base64.b64encode(svg_bytes).decode('utf-8')}"
            images.append(dict(
                source=uri,
                xref="x", yref="y",
                x=point.x,
                y=7-point.y,
                sizex=1.0, sizey=1.0,
                xanchor="center", yanchor="middle",
                layer="above"
            ))
    return images

def create_board_figure(board: ChessBoardArray, mask: np.array):
    # Create base array for heatmap
    z = np.flipud(np.reshape(mask, (8,8)))

    # Create transparent red heatmap as mask
    heatmap = go.Heatmap(
        z=z,
        x=list(range(8)),
        y=list(range(8)),
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(255,0,0,0.4)']],
        zmin=0,
        zmax=1,
        showscale=False,
        hoverinfo='skip',
        xgap=0,
        ygap=0,
        opacity=1.0,
    )

    fig = go.Figure(data=[heatmap])

    fig.update_layout(
        width=320,
        height=320,
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(
            scaleanchor="x",
            scaleratio=1,
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-0.5, 7.5],
            fixedrange=True
        ),
        xaxis=dict(
            constrain='domain',
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-0.5, 7.5],
            fixedrange=True
        ),
        images=board_to_svg_image(board),
        shapes=[],
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    return fig

def create_dual_board_app(boards1: list[ChessBoardArray], 
                          boards2: list[ChessBoardArray], 
                          # boards3: list[ChessBoardArray],
                          # boards4: list[ChessBoardArray],
                          masks):
    assert len(boards1) == len(boards2), "Board lists must be the same length"

    num_steps = len(boards1)
    app = dash.Dash(__name__)

    app.layout = html.Div([
        html.Div([
            html.Button("Prev", id="prev-btn", n_clicks=0),
            html.Button("Next", id="next-btn", n_clicks=0),
            html.Span(id="step-label", style={"marginLeft": "1rem"}),
        ], style={"marginBottom": "1rem"}),

        dcc.Store(id="current-step", data=0),

        html.Div([
            dcc.Graph(id='left-board', config={"displayModeBar": False}),
            dcc.Graph(id='right-board', config={"displayModeBar": False}),
        ], style={"display": "flex", "justifyContent": "center", "gap": "2rem"}),
    ])

    @app.callback(
        Output('current-step', 'data'),
        Output('step-label', 'children'),
        Input('prev-btn', 'n_clicks'),
        Input('next-btn', 'n_clicks'),
        State('current-step', 'data')
    )
    def update_step(prev, nxt, current):
        ctx = dash.callback_context.triggered_id
        if ctx == 'prev-btn':
            current = max(0, current - 1)
        elif ctx == 'next-btn':
            current = min(num_steps - 1, current + 1)
        return current, f"Step: {current} / {num_steps - 1}"

    @app.callback(
        Output('left-board', 'figure'),
        Output('right-board', 'figure'),
        Input('current-step', 'data')
    )
    def update_boards(idx):
        mask = masks[idx]
        return (
            create_board_figure(boards1[idx], mask),
            create_board_figure(boards2[idx], mask),
        )

    return app

In [ ]:
pgn = PGNBoardHelper(Path('/Users/ray/Datasets/chess/Carlsen.pgn'))

ckpt = torch.load('/Users/ray/models/chess/transformer/0eb35bb7-5c81-4f8f-829a-f00cd1686ac7/last.ckpt', map_location="cpu")
model = ChessTransformer(**ckpt['hyper_parameters'])
model.load_state_dict(ckpt['state_dict'])

for i in range(1729):
    pgn.get_game()

tokenizer = SimpleChessTokenizer()
game_batch = pgn.get_game_batch(tokenizer=tokenizer)

In [6]:
helper = MaskVisualizationHelper(transformer=model)
m, current, nxt = helper.get_preds(game_batch)

In [7]:
current_pred = [ChessBoardArray(arr) for arr in current.numpy()]
current_label = [ChessBoardArray(arr) for arr in game_batch['board'].squeeze().numpy()]
nxt_pred = [ChessBoardArray(arr) for arr in nxt.numpy()]
nxt_label = [ChessBoardArray(arr) for arr in game_batch['next_board'].squeeze().numpy()]

In [8]:
app = create_dual_board_app(current_label, current_pred, m)
app.run(mode="jupyter-inline", port=8051)